# Sentence-boundary cut exploration

Compares the current word-level extraction pipeline (`../remove_lexical_confounds.ipynb`, already
deployed to `CoT_datasets/lexically_cleaned/`) against the alternative sentence-boundary cut
(remove the whole final-judgement sentence instead of cutting at a single disclosure word),
across two shift/balance designs:

- **v1 (current production methodology)**: shift *any* perfectly-monotone landing word
  regardless of count, then balance every resulting word to `min(true, false)`.
- **v2 (corrected)**: shift a monotone word only if it occurs *more* than `MIN_COUNT` times
  (large sample -> likely a generic framing word, safe to relocate); leave any word (monotone
  or not) with count `<= MIN_COUNT` completely untouched -- forcing a small bucket through
  balance would drop it to zero (a monotone bucket's `min(true,false)` is 0 regardless of
  size), which unfairly punishes rare, likely-informative words (specific computed values,
  place names) that occur too rarely to teach a linear probe a reliable shortcut anyway.

**Shift/balance decisions are derived per-task from the TRAIN split only, then the same
`shift_words` set is applied to both that task's train and test rows.** An earlier version of
this notebook computed monotonicity independently per (task, split) -- which meant a split's
own labels were used to decide how that same split gets transformed, a real train/test leakage
bug, distinct from the (smaller) issue where a common word split thinly across groups could
dodge the shift threshold everywhere. Deciding from train only and applying to test fixes both.

The sentence-boundary cut itself (`sentence_start_before`) mirrors `remove_last_sentence` in
`sentence_boundary_validation_1800.py` exactly (naive last-`.` search) -- validated as the best
of the boundary-detection variants tried; see that script and the surrounding investigation.

Run top to bottom -- this is read-only exploration, it does not write to
`CoT_datasets/filtered/` or `CoT_datasets/lexically_cleaned/`.

In [1]:
import re

import pandas as pd

FILTERED_DIR = "../../CoT_datasets/filtered"
RAW_LABELS_DIR = "../../dataset"
TASKS = ["F0", "F1", "F2", "F3", "F4", "F5", "A1", "A2", "A3"]
SPLITS = ["train", "test"]
MIN_COUNT = 11


def get_statement(generated_text: str) -> str:
    return generated_text.split("<｜User｜>")[1].split("\n")[0].strip()


def sentence_start_before(text: str) -> int:
    """Mirrors `remove_last_sentence` in sentence_boundary_validation_1800.py exactly: find
    the very last literal '.' anywhere in the text (no whitespace-after or closing-punctuation
    requirement) and cut right after it. Validated via a 1800-sample Azure batch (~1.9%
    information-loss rate) and compared against two whitespace-aware variants that each
    regressed MORE rows (33-141) than the rare bug (decimal points / mid-sentence truncation)
    they were meant to fix -- this naive version is the one actually used in production, so
    this notebook should reflect it rather than a different, unvalidated regex."""
    text = text.rstrip()
    pos = text.rfind(".")
    return (pos + 1) if pos != -1 else 0


def last_two_words(text: str):
    # includes bare digit sequences as words -- otherwise a sentence ending in a number
    # (e.g. "...is also 0.") gets silently mis-bucketed under whatever letter-word precedes
    # the number instead of the number itself, hiding ~19% of rows from this whole analysis
    words = re.findall(r"[A-Za-z']+|\d+", text.rstrip())
    if len(words) < 2:
        return None, (words[-1].lower() if words else None)
    return words[-2].lower(), words[-1].lower()

## 1. Load all tasks, recover labels, compute both landing-word representations

In [2]:
def load_task(task: str, split: str) -> pd.DataFrame:
    filt = pd.read_csv(f"{FILTERED_DIR}/{task}_filtered_{split}.csv")
    raw = pd.read_csv(f"{RAW_LABELS_DIR}/{task}_{split}.csv")[["statement", "label"]].drop_duplicates(subset="statement")

    filt = filt.copy()
    filt["statement"] = filt["generated_statement_texts"].apply(get_statement)
    merged = filt.merge(raw, on="statement", how="left").dropna(subset=["label"])
    merged["label"] = merged["label"].astype(bool)

    # word-level cut (current methodology): landing word = last word of the current extraction
    merged[["word_prev_w", "word_last_w"]] = merged["extracted_statement_texts"].apply(
        lambda t: pd.Series(last_two_words(t))
    )

    # sentence-boundary cut (alternative): back up to the start of the sentence containing
    # the current cut point, landing word = whatever now ends the kept text
    merged["sent_idx"] = merged["extracted_statement_texts"].apply(sentence_start_before)
    merged["sent_text"] = merged.apply(lambda r: r["extracted_statement_texts"][:r["sent_idx"]].rstrip(), axis=1)
    merged[["sent_prev_w", "sent_last_w"]] = merged["sent_text"].apply(lambda t: pd.Series(last_two_words(t)))

    merged["task"] = task
    merged["split"] = split
    return merged


data = {(task, split): load_task(task, split) for task in TASKS for split in SPLITS}
print(f"Loaded {len(data)} task/split combinations, {sum(len(d) for d in data.values())} total rows")

Loaded 18 task/split combinations, 14049 total rows


## 2. Shift/balance functions -- shift set computed from TRAIN only, applied to any split

In [3]:
def compute_shift_v1(train_df: pd.DataFrame, last_col: str):
    """Existing production methodology: any perfectly-monotone word (computed from TRAIN
    only) gets shifted, regardless of count. No count-based protection."""
    counts = train_df[last_col].value_counts()
    shift_words = set()
    for w in counts.index:
        sub = train_df[train_df[last_col] == w]
        t = sub["label"].sum()
        if t == 0 or t == len(sub):
            shift_words.add(w)
    return shift_words


def compute_shift_v2(train_df: pd.DataFrame, last_col: str, min_count: int = MIN_COUNT):
    """Shift a monotone word (computed from TRAIN only) only if it occurs MORE than
    min_count times there (large sample -- likely generic framing, safe to relocate). Words
    with count <= min_count are left for apply_shift_and_balance to keep whole rather than
    balance -- forcing a small bucket through balance would drop it to zero regardless of
    size, unfairly punishing rare, likely-informative words (specific computed values, place
    names) that occur too rarely to teach a linear probe a reliable shortcut anyway.

    Returns (shift_words, shift_info): shift_words is bare words for the membership check in
    apply_shift_and_balance; shift_info is (word, count) pairs, kept separate purely so
    callers can inspect what got shifted -- `x in shift_set` would silently never match
    anything once shift_set holds tuples instead of bare strings."""
    counts = train_df[last_col].value_counts()
    shift_words = set()
    shift_info = set()
    for w in counts.index:
        sub = train_df[train_df[last_col] == w]
        t = sub["label"].sum()
        n = len(sub)
        if (t == 0 or t == n) and n > min_count:
            shift_words.add(w)
            shift_info.add((w, n))
    return shift_words, shift_info


def apply_shift_and_balance(df: pd.DataFrame, prev_col: str, last_col: str, shift_words: set, min_count: int = None):
    """Apply a precomputed shift set (from compute_shift_v1/v2, always derived from that
    task's TRAIN split) to df -- which may be the train split itself or the held-out test
    split -- then balance every resulting bucket to min(true, false). If min_count is given,
    any bucket with <= min_count rows is kept whole instead of balanced (v2's small-bucket
    protection)."""
    d = df.copy()
    d["final_w"] = d.apply(lambda r: r[prev_col] if r[last_col] in shift_words else r[last_col], axis=1)

    kept = 0
    for w, g in d.groupby("final_w", dropna=False):
        if min_count is not None and len(g) <= min_count:
            kept += len(g)  # small bucket: keep everything, don't balance
        else:
            t = int(g["label"].sum()); f = len(g) - t
            kept += 2 * min(t, f)
    return kept

## 3. Run the comparison across all tasks/splits

In [4]:
rows = []
shift_words_sent = set()
shift_words_word = set()
overall = {"word_v1_current": [0, 0], "word_v2_corrected": [0, 0], "sentence_v2_corrected": [0, 0]}

for task in TASKS:
    train = data[(task, "train")]
    test = data[(task, "test")]

    word_v1_shift = compute_shift_v1(train, "word_last_w")
    word_v2_shift, word_v2_info = compute_shift_v2(train, "word_last_w")
    sent_v2_shift, sent_v2_info = compute_shift_v2(train, "sent_last_w")
    shift_words_word |= word_v2_info
    shift_words_sent |= sent_v2_info

    for split, df in [("train", train), ("test", test)]:
        n = len(df)
        word_v1_kept = apply_shift_and_balance(df, "word_prev_w", "word_last_w", word_v1_shift)
        word_v2_kept = apply_shift_and_balance(df, "word_prev_w", "word_last_w", word_v2_shift, MIN_COUNT)
        sent_v2_kept = apply_shift_and_balance(df, "sent_prev_w", "sent_last_w", sent_v2_shift, MIN_COUNT)

        rows.append({
            "task": task, "split": split, "n": n,
            "word_v1_current_%": round(100 * word_v1_kept / n, 1),
            "word_v2_corrected_%": round(100 * word_v2_kept / n, 1),
            "sentence_v2_corrected_%": round(100 * sent_v2_kept / n, 1),
        })

        for name, kept in [("word_v1_current", word_v1_kept), ("word_v2_corrected", word_v2_kept), ("sentence_v2_corrected", sent_v2_kept)]:
            overall[name][0] += kept
            overall[name][1] += n

results = pd.DataFrame(rows)
results

,task,split,n,word_v1_current_%,word_v2_corrected_%,sentence_v2_corrected_%
0,F0,train,1187,67.1,73.5,84.2
1,F0,test,512,66.8,76.8,90.6
2,F1,train,1193,81.5,85.3,92.2
3,F1,test,511,78.7,89.8,96.1
4,F2,train,1194,41.4,49.7,86.5
5,F2,test,509,38.5,53.8,91.2
6,F3,train,1394,78.0,80.3,72.3
7,F3,test,598,76.3,80.8,79.9
8,F4,train,1392,70.5,73.0,81.5
9,F4,test,598,80.6,83.3,95.0


In [5]:
print("OVERALL retention across all 9 tasks (shift/balance decisions derived from TRAIN only, applied to both splits):")
for name, (kept, n) in overall.items():
    print(f"  {name:25s}: {kept}/{n} ({100*kept/n:.1f}%)")

OVERALL retention across all 9 tasks (shift/balance decisions derived from TRAIN only, applied to both splits):
  word_v1_current          : 8310/14049 (59.2%)
  word_v2_corrected        : 9421/14049 (67.1%)
  sentence_v2_corrected    : 12441/14049 (88.6%)


## 4. Words that get shifted under the corrected (v2) design -- worth a manual look

In [6]:
print(f"Word-level cut: {len(shift_words_word)} words shifted (monotone, n>{MIN_COUNT})")
print(sorted(shift_words_word))
print()
print(f"Sentence-boundary cut: {len(shift_words_sent)} words shifted (monotone, n>{MIN_COUNT})")
print(sorted(shift_words_sent))

Word-level cut: 18 words shifted (monotone, n>11)
[('checks', 14), ('checks', 20), ("doesn't", 17), ("doesn't", 23), ("doesn't", 164), ("doesn't", 177), ('holds', 14), ('holds', 17), ('holds', 18), ('holds', 37), ('holds', 93), ("isn't", 12), ("isn't", 14), ("isn't", 29), ('not', 16), ('something', 14), ("there's", 65), ('which', 62)]

Sentence-boundary cut: 5 words shifted (monotone, n>11)
[('condition', 41), ('is', 16), ('one', 16), ('otherwise', 15), ('out', 13)]
